In [1]:
# Imports
import pandas as pd
import numpy as np
import re
import pickle
from collections import Counter

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")

All libraries imported successfully.
PyTorch version : 2.11.0+cpu


In [2]:
# Load Dataset 
train = pd.read_csv('../data/sent_train.csv')
valid = pd.read_csv('../data/sent_valid.csv')

print(f"Train shape : {train.shape}")
print(f"Valid shape : {valid.shape}")

Train shape : (9543, 2)
Valid shape : (2388, 2)


In [3]:
# Text Cleaning Function
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))
# Remove financially meaningful words from stopwords
keep_words = {'up', 'down', 'not', 'no', 'nor', 'against', 'below', 'above', 'under', 'over'}
stop_words = stop_words - keep_words

def clean_text(text):
    # Remove
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)  #URL
    text = re.sub(r'@\w+', '', text)  #@mentions
    text = re.sub(r'\$([A-Za-z]+)', r'\1', text)   #$
    text = re.sub(r'#(\w+)', r'\1', text)  #hashtags
    text = re.sub(r'[^a-z\s]', '', text)   #spl char, punc
    text = re.sub(r'\s+', ' ', text).strip()   #whitespace
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]   #topwords and short tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Test on sample tweets
samples = [
    "$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/abc123",
    "$TSLA - Tesla beats Q4 estimates, raises guidance @elonmusk #Tesla",
    "Central banks don't have as much monetary policy power as they used to."
]

print("CLEANING TEST:")
print("-" * 60)
for s in samples:
    print(f"Original : {s}")
    print(f"Cleaned  : {clean_text(s)}")
    print()

CLEANING TEST:
------------------------------------------------------------
Original : $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/abc123
Cleaned  : bynd jpmorgan reel expectation beyond meat

Original : $TSLA - Tesla beats Q4 estimates, raises guidance @elonmusk #Tesla
Cleaned  : tsla tesla beat estimate raise guidance tesla

Original : Central banks don't have as much monetary policy power as they used to.
Cleaned  : central bank dont much monetary policy power used



In [4]:
# Apply Cleaning
train['cleaned_text'] = train['text'].apply(clean_text)
valid['cleaned_text'] = valid['text'].apply(clean_text)

# Verify no empty strings after cleaning
train_empty = (train['cleaned_text'].str.strip() == '').sum()
valid_empty  = (valid['cleaned_text'].str.strip() == '').sum()

print(f"Empty texts after cleaning — Train : {train_empty}")
print(f"Empty texts after cleaning — Valid : {valid_empty}")
print()
print("Sample cleaned texts (Train):")
print("-" * 60)
for i in range(5):
    print(f"Original : {train['text'].iloc[i]}")
    print(f"Cleaned  : {train['cleaned_text'].iloc[i]}")
    print(f"Label    : {train['label'].iloc[i]}")
    print()

Empty texts after cleaning — Train : 8
Empty texts after cleaning — Valid : 0

Sample cleaned texts (Train):
------------------------------------------------------------
Original : $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Cleaned  : bynd jpmorgan reel expectation beyond meat
Label    : 0

Original : $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3
Cleaned  : ccl rcl nomura point booking weakness carnival royal caribbean
Label    : 0

Original : $CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb
Cleaned  : cemex cut credit suisse morgan weak building outlook
Label    : 0

Original : $ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N
Cleaned  : es btig research cut neutral
Label    : 0

Original : $FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB
Cleaned  : fnko funko slide piper jaffray cut
Label    : 0



In [5]:
# Handle Empty Texts
print("Empty texts in Train:")
print(train[train['cleaned_text'].str.strip() == ''][['text', 'label']])
print()

# Drop empty cleaned texts from train
train = train[train['cleaned_text'].str.strip() != ''].reset_index(drop=True)

print(f"Train shape after dropping empty texts : {train.shape}")
print(f"Valid shape unchanged                  : {valid.shape}")
print()

# Verify class distribution is not affected significantly
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
print("Train class distribution after cleaning:")
for label_id, label_name in label_map.items():
    count = (train['label'] == label_id).sum()
    pct   = count / len(train) * 100
    print(f"  {label_name:<10} : {count:>5}  ({pct:.1f}%)")

Empty texts in Train:
                                                   text  label
3943                                                 :)      2
3948                                            @TicToc      2
3949  @tictoc @telefenoticias @teleSUR_Chile @Paolad...      2
3950  @tictoc @telefenoticias @teleSUR_Chile @Paolad...      2
4440                                                 F5      2
4681                            https://t.co/575AH1YRkF      2
4682                            https://t.co/9eZPvQhfMq      2
4683                            https://t.co/oJxNPEUpWq      2

Train shape after dropping empty texts : (9535, 3)
Valid shape unchanged                  : (2388, 3)

Train class distribution after cleaning:
  Bearish    :  1442  (15.1%)
  Bullish    :  1923  (20.2%)
  Neutral    :  6170  (64.7%)


In [6]:
# Build Vocabulary

PAD_TOKEN   = '<PAD>'   # padding token
UNK_TOKEN   = '<UNK>'   # unknown token for words not in vocabulary
MIN_FREQ    = 2         # ignore words appearing less than 2 times

# Count word frequencies in train
word_freq = Counter()
for text in train['cleaned_text']:
    word_freq.update(text.split())

# Filter by minimum frequency
vocab_words = [w for w, freq in word_freq.items() if freq >= MIN_FREQ]

# Build word2idx — reserve 0 for PAD, 1 for UNK
word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for word in sorted(vocab_words):
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

VOCAB_SIZE = len(word2idx)

print(f"Total unique words in train  : {len(word_freq):,}")
print(f"Words with freq >= {MIN_FREQ}       : {len(vocab_words):,}")
print(f"Vocabulary size (with PAD+UNK): {VOCAB_SIZE:,}")
print(f"PAD index : {word2idx[PAD_TOKEN]}")
print(f"UNK index : {word2idx[UNK_TOKEN]}")

Total unique words in train  : 14,060
Words with freq >= 2       : 6,304
Vocabulary size (with PAD+UNK): 6,306
PAD index : 0
UNK index : 1


In [7]:
#Text to Sequence 
MAX_SEQUENCE_LENGTH = 32  # confirmed from EDA — covers 100% of data

def text_to_sequence(text, word2idx, max_len):
    tokens = text.split()
    # Map tokens 
    sequence = [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]
    # Truncate if longer 
    sequence = sequence[:max_len]
    # Pad if shorter 
    sequence = sequence + [word2idx[PAD_TOKEN]] * (max_len - len(sequence))
    return sequence

#Test
sample_text = train['cleaned_text'].iloc[0]
sample_seq  = text_to_sequence(sample_text, word2idx, MAX_SEQUENCE_LENGTH)

print(f"Sample cleaned text : {sample_text}")
print(f"Sequence length     : {len(sample_seq)}")
print(f"Sequence            : {sample_seq}")
print()

# Verify padding
short_text = "stock down"
short_seq  = text_to_sequence(short_text, word2idx, MAX_SEQUENCE_LENGTH)
print(f"Short text          : '{short_text}'")
print(f"Sequence length     : {len(short_seq)}")
print(f"Padded sequence     : {short_seq}")

Sample cleaned text : bynd jpmorgan reel expectation beyond meat
Sequence length     : 32
Sequence            : [847, 2974, 4514, 1967, 610, 3410, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Short text          : 'stock down'
Sequence length     : 32
Padded sequence     : [5326, 1659, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [8]:
#Convert Full Dataset to Tensors 
def dataset_to_tensors(df, word2idx, max_len):
    sequences = [text_to_sequence(text, word2idx, max_len) for text in df['cleaned_text']]
    X = torch.tensor(sequences, dtype=torch.long)
    y = torch.tensor(df['label'].values, dtype=torch.long)
    return X, y

X_train, y_train = dataset_to_tensors(train, word2idx, MAX_SEQUENCE_LENGTH)
X_valid, y_valid = dataset_to_tensors(valid, word2idx, MAX_SEQUENCE_LENGTH)

print(f"X_train shape : {X_train.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"X_valid shape : {X_valid.shape}")
print(f"y_valid shape : {y_valid.shape}")
print()
print(f"X_train dtype : {X_train.dtype}")
print(f"y_train dtype : {y_train.dtype}")
print()
print(f"y_train class counts : {torch.bincount(y_train)}")
print(f"y_valid class counts : {torch.bincount(y_valid)}")

X_train shape : torch.Size([9535, 32])
y_train shape : torch.Size([9535])
X_valid shape : torch.Size([2388, 32])
y_valid shape : torch.Size([2388])

X_train dtype : torch.int64
y_train dtype : torch.int64

y_train class counts : tensor([1442, 1923, 6170])
y_valid class counts : tensor([ 347,  475, 1566])
